## **Protocol 002 — Stage 2**: pLDDT Values and Secondary Structure Analysis

**Input:**
- AlphaFold output folders for control and toxin signal peptides,
  each containing a `result_model_1_pred_0.pkl` file with per-residue pLDDT scores.
- PDB files of signal peptide models (used as input for STRIDE).

**Output:**
- `plddtvector_control_AF.xlsx` — per-residue pLDDT vectors for control SPs (padded to length 69).
- `dict_toxin_plddt_Figure4.xlsx` — pLDDT confidence category proportions for toxin SPs.
- STRIDE secondary structure assignment files (one `.pdb` output per model).
- Residue counts per secondary structure type (helix, coil, turn, beta-sheet).

### Install Requirements

In [38]:
%pip install biopandas
from biopandas.pdb import PandasPdb
import pandas as pd
import numpy as np
import shutil
import os
import pickle

Note: you may need to restart the kernel to use updated packages.


### Helper Functions

Three utility functions are defined to support pLDDT score processing:
- `cate_color`: bins per-residue pLDDT scores into four AlphaFold confidence tiers.
- `addna`: pads a pLDDT list to length 42, centering the original values (used for heatmap alignment).
- `addna_ceros`: pads a pLDDT list to length 69 by appending zeros (used for per-SP vectors).

In [4]:
def cate_color(plddt):
    """
    Categorize per-residue pLDDT scores into four AlphaFold confidence tiers.

    Args:
        plddt (list): List of per-residue pLDDT scores (0–100).

    Returns:
        list: Integer category per residue:
              1 = Very high (pLDDT >= 90)
              2 = Confident (70 <= pLDDT < 90)
              3 = Low       (50 <= pLDDT < 70)
              4 = Very low  (pLDDT < 50)
    """
    color_list = []

    for i in plddt:
        if i >= 90:
            color_list.append(1)   # Very high confidence
        elif i >= 70 and i < 90:
            color_list.append(2)   # Confident
        elif i >= 50 and i < 70:
            color_list.append(3)   # Low confidence
        elif i < 50:
            color_list.append(4)   # Very low confidence
    return color_list


def addna(lista):
    """
    Pad a pLDDT list to a fixed length of 42 by centering the original values
    and filling both sides with zeros. Appends the original length at the end.

    Args:
        lista (list): Per-residue pLDDT or category values.

    Returns:
        list: Zero-padded list of length 43 (42 values + original length).
    """
    longitud = len(lista)
    sobrante = 42 - longitud
    listanew = []

    if sobrante % 2 == 0:
        mitad = int(sobrante / 2)
        listanew = mitad * [0] + list(lista) + mitad * [0]
    else:
        mitad2 = int(sobrante / 2)
        listanew = mitad2 * [0] + list(lista) + (mitad2 + 1) * [0]

    return listanew + [longitud]  # Append original length as last element


def addna_ceros(lista):
    """
    Pad a pLDDT list to a fixed length of 69 by appending zeros.
    Used to create uniform-length vectors for all signal peptides.

    Args:
        lista (list): Per-residue pLDDT values.

    Returns:
        list: Zero-padded list of length 69.
    """
    longitud = len(lista)
    sobrante = 69 - longitud
    return list(lista) + int(sobrante) * [0]


# Quick test
print(cate_color([90, 88]))
print(addna([2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1]))

[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 19]


---
## Part 1: pLDDT Score Analysis from AlphaFold Predictions

Per-residue pLDDT scores are extracted from AlphaFold's pickle output files
(`result_model_1_pred_0.pkl`). Each file is loaded and the `plddt` array is
extracted and padded to a uniform length for downstream analysis.

> **Note:** Update `path` variables below to point to your local AlphaFold output directories.

### 1.1 Control Signal Peptides — Extract pLDDT Vectors

In [10]:
import numpy as np
import os

matrix = []

# Path to the folder containing AlphaFold output subfolders for control SPs
# Each subfolder should contain: sequence/result_model_1_pred_0.pkl
path = "data/2_AlphaFold_Control_SPs_PDBs"  # Update to your local path

with os.scandir(path) as entries:
    for entry in entries:
        # Load the AlphaFold pickle file for each signal peptide
        pkl_path = os.path.join(path, entry.name, 'sequence', 'result_model_1_pred_0.pkl')
        with open(pkl_path, 'rb') as pkl_file:
            pkl_result = pickle.load(pkl_file)

            # Pad pLDDT vector to length 69 and prepend the folder name
            row = np.concatenate(
                (np.array([entry.name]), addna_ceros(pkl_result['plddt'])),
                axis=None
            )
            matrix.append(row)

# Save control SP pLDDT vectors to Excel
dc = pd.DataFrame(data=matrix)
dc.to_excel('plddtvector_control_AF.xlsx')
print(f"Control SPs processed: {len(matrix)}")

### 1.2 Toxin Signal Peptides — Aggregate pLDDT Scores

In [40]:
import numpy as np
import os

matrix = []

# Path to the folder containing AlphaFold output subfolders for toxin SPs
path = "data/2_AlphaFold_Toxin_SPs_PDBs"  # Update to your local path

with os.scandir(path) as entries:
    for entry in entries:
        # Load the AlphaFold pickle file for each signal peptide
        pkl_path = os.path.join(path, entry.name, 'sequence', 'result_model_1_pred_0.pkl')
        with open(pkl_path, 'rb') as pkl_file:
            pkl_result = pickle.load(pkl_file)

            # Concatenate all pLDDT values into a single flat array across all toxin SPs
            matrix = np.concatenate((matrix, pkl_result['plddt']), axis=None)

# Count residues per confidence tier across all toxin SPs
cvh = sum(1 for i in matrix if i >= 90)           # Very high
ch  = sum(1 for i in matrix if 70 <= i < 90)      # Confident
cl  = sum(1 for i in matrix if 50 <= i < 70)      # Low
cvl = sum(1 for i in matrix if i < 50)            # Very low

print(f"Very high (>=90): {cvh}")
print(f"Confident (70-90): {ch}")
print(f"Low (50-70): {cl}")
print(f"Very low (<50): {cvl}")
print(f"Total residues: {cvh + ch + cl + cvl}")

3644 14584 10019 581


Inspect an example row from the pLDDT matrix to verify the padded vector format:

In [22]:
# Display the second entry as a sanity check
matrix[1]

array(['002_D5CBA0', '57.72947577275044', '64.81869691517748',
       '63.18877751682885', '65.58127013633201', '65.10157278615952',
       '68.29925986697344', '73.74643842588192', '80.83079487129042',
       '83.89563270537292', '85.11413797964632', '88.06292621949068',
       '88.14529452356507', '90.14920811295495', '90.38819540423991',
       '90.25150712300477', '91.1350930133045', '90.37191757683054',
       '88.43145937321259', '88.81891032234329', '88.23162275459654',
       '85.09797412006822', '85.08980080157275', '85.31740762156343',
       '80.01990997843177', '68.89969497546552', '66.39841350094137',
       '64.26957549163852', '59.99937712151039', '62.8319404475951',
       '61.63327736152313', '63.27434632205949', '58.573050638660774',
       '0.0', '0.0', '0.0', '0.0', '0.0', '0.0', '0.0', '0.0', '0.0',
       '0.0', '0.0', '0.0', '0.0', '0.0', '0.0', '0.0', '0.0', '0.0',
       '0.0', '0.0', '0.0', '0.0', '0.0', '0.0', '0.0', '0.0', '0.0',
       '0.0', '0.0', '0.0', 

### 1.3 Confidence Tier Proportions for Toxin SP Database

Calculate the proportion of residues in each pLDDT confidence tier
across all 917 toxin signal peptide models (28,828 residues total).
These proportions are used in Figure 2B of the manuscript.

In [5]:
# Build a dictionary mapping SP name → pLDDT list
# (dict_toxin_plddt is populated from the pkl files in Step 1.2)
names = list(dict_toxin_plddt.keys())
plddt = list(dict_toxin_plddt.values())

# Lambda to compute fraction of values within a confidence range
get_freq = lambda a, low, high: sum(low <= i < high for i in a) / len(a)

a = get_freq(plddt, 90, 100)   # Very high
b = get_freq(plddt, 70, 90)    # Confident
c = get_freq(plddt, 50, 70)    # Low
d = get_freq(plddt, 0, 50)     # Very low

plddt_per = [a, b, c, d]
print("pLDDT proportions [very high, confident, low, very low]:", plddt_per)

NameError: name 'dict_toxin_plddt' is not defined

### 1.4 Save pLDDT Category Data to Excel

Export the per-residue confidence category matrix for all toxin SPs.
Output used to generate Figure 2B.

In [7]:
# Build a DataFrame from the toxin pLDDT dictionary (SP name → pLDDT values)
df = pd.DataFrame(data=dict_toxin_plddt)
df = df.T  # Transpose: rows = SPs, columns = residue positions
df.to_excel('dict_toxin_plddt_Figure4.xlsx')
print("Saved: dict_toxin_plddt_Figure4.xlsx")

---
## Part 2: Secondary Structure Analysis (STRIDE)

The **STRIDE** algorithm assigns secondary structure (alpha-helix, beta-sheet,
coil, turn) to each residue from PDB coordinate files.
STRIDE is run on all signal peptide PDB models, and the residue-level
assignments are counted to characterize the structural composition of the SP databases.

> **Note:** STRIDE must be compiled locally. Source available at:
> http://webclu.bio.wzw.tum.de/stride/

### 2.1 Compile STRIDE from Source

In [28]:
# Navigate to the STRIDE source directory and compile
# Update this path to where the STRIDE source code is located
%cd /path/to/stride
!make

C:\Users\joc_t\Documents\2022\Congreso 2022 Paper\Version 2\stride
gcc -O2   stride.o splitstr.o rdpdb.o initchn.o geometry.o thr2one.o one2thr.o filename.o tolostr.o strutil.o place_h.o hbenergy.o memory.o helix.o sheet.o rdmap.o phipsi.o command.o molscr.o die.o hydrbond.o mergepat.o fillasn.o escape.o p_jrnl.o p_rem.o p_atom.o p_helix.o p_sheet.o p_turn.o p_ssbond.o p_expdta.o p_model.o p_compnd.o report.o nsc.o area.o ssbond.o chk_res.o chk_atom.o turn.o pdbasn.o dssp.o outseq.o chkchain.o elem.o measure.o asngener.o p_endmdl.o stred.o contact_order.o contact_map.o  -lm  -o  ./stride


### 2.2 Run STRIDE on All Signal Peptide PDB Files

STRIDE is applied to each PDB model in the toxin SP database.
Each output file contains the secondary structure assignment per residue.

In [57]:
# Path to the folder containing input PDB files for toxin SPs
path_pdb = "data/2.1Input_PDBs_for_STRIDE"  # Update to your local path
path_out = "data/2.1_STRIDE_SecondaryStructure_results"  # Output folder
os.makedirs(path_out, exist_ok=True)

# Run STRIDE on each PDB file and write output to the results folder
with os.scandir(path_pdb) as entries:
    for entry in entries:
        if entry.name.endswith('.pdb'):
            !stride {path_pdb}/{entry.name} > {path_out}/{entry.name}

C:\Users\joc_t\Documents\2022\Congreso 2022 Paper\Version 2\stride


Error reading PDB file toxin_full_pdb/TEST.txt


Inspect STRIDE output for a single PDB file to verify the assignment format:

In [53]:
# Example: run STRIDE on a single PDB and display the output
!stride data/2.1Input_PDBs_for_STRIDE/002_D5CBA0.pdb

REM  -------------------- Secondary structure summary -------------------  ~~~~
REM                                                                        ~~~~
CHN  toxin_full_pdb/002_D5CBA0.pdb A                                       ~~~~
REM                                                                        ~~~~
REM                .         .         .                                   ~~~~
SEQ  1    MMKQDQVRFSQRALSALLSVLLATQPLLPAVA                     32          ~~~~
STR        HHHHHHHHHHHHHHHHHHHHHHHTTTGGG                                   ~~~~
REM                                                                        ~~~~
REM                                                                        ~~~~
REM                                                                        ~~~~
LOC  AlphaHelix   MET     2 A      THR     24 A                            ~~~~
LOC  310Helix     LEU    28 A      ALA     30 A                            ~~~~
LOC  TurnI        THR    24 A      LEU  

### 2.3 Count Residues per Secondary Structure Type

Parse all STRIDE output files and count residues assigned to each
secondary structure class. STRIDE codes used here:
- `H` = Alpha-helix
- `E` = Extended beta-strand
- `B`/`b` = Isolated beta-bridge
- `T` = Turn
- `C` = Coil (random coil)
- `G` = 3/10-helix
- `I` = Pi-helix

In [24]:
import os

# Path to the folder containing STRIDE output files
path = "data/2.1_STRIDE_SecondaryStructure_results"  # Update to your local path

# Initialize residue counters for each secondary structure type
nH = 0   # Alpha-helix
nE = 0   # Extended beta-strand
nB = 0   # Isolated beta-bridge (capital B)
nb = 0   # Isolated beta-bridge (lowercase b)
nT = 0   # Turn
nC = 0   # Coil
nG = 0   # 3/10-helix
nI = 0   # Pi-helix

with os.scandir(path) as entries:
    for entry in entries:
        with open(os.path.join(path, entry.name), 'r') as f:
            for line in f.readlines():
                # ASG lines contain per-residue secondary structure assignments
                if 'ASG' in line:
                    if ' H ' in line: nH += 1
                    elif ' E ' in line: nE += 1
                    elif ' B ' in line: nB += 1
                    elif ' b ' in line: nb += 1
                    elif ' T ' in line: nT += 1
                    elif ' C ' in line: nC += 1
                    elif ' G ' in line: nG += 1
                    elif ' I ' in line: nI += 1

print(f"Alpha-helix (H):         {nH}")
print(f"Beta-strand (E):         {nE}")
print(f"Beta-bridge (B+b):       {nB + nb}")
print(f"Turn (T):                {nT}")
print(f"Coil (C):                {nC}")
print(f"3/10-helix (G):          {nG}")
print(f"Pi-helix (I):            {nI}")
print(f"Total residues:          {nH+nE+nB+nb+nT+nC+nG+nI}")

18416 788 4 0 3483 5642 524 0


Verify coil assignments for a single file to confirm the parser is working correctly:

In [13]:
# Inspect all Coil-assigned residues in a single STRIDE output file
with open(path + '/001_A0A1S4NYE3.pdb', 'r') as f:
    for line in f.readlines():
        if 'ASG' in line and 'Coil' in line:
            print(line.strip())

111111 ASG  MET A    1    1    C          Coil    360.00     81.80     234.6      ~~~~

111111 ASG  HIS A    2    2    C          Coil    -97.99     90.40     195.8      ~~~~

111111 ASG  GLN A    3    3    C          Coil    -71.35    122.64     179.5      ~~~~

111111 ASG  PRO A    4    4    C          Coil    -58.39    133.96     109.2      ~~~~

111111 ASG  PRO A    5    5    C          Coil    -55.88    121.53     116.3      ~~~~

111111 ASG  VAL A    6    6    C          Coil    -51.53    109.90     117.7      ~~~~

111111 ASG  ARG A    7    7    C          Coil    -56.35    127.51     176.8      ~~~~

111111 ASG  ALA A   32   32    C          Coil   -106.25    360.00     141.0      ~~~~



### 2.4 Summary: Total Residue Counts

Cross-check the total residue counts from secondary structure and pLDDT analyses.
Both should sum to the same total (28,828 residues from 917 toxin SP models).

In [25]:
# Total residues from secondary structure analysis
total_ss = 18416 + 788 + 4 + 0 + 3483 + 5642 + 524 + 0
print(f"Total residues (STRIDE): {total_ss}")

28857


In [26]:
# Total residues from pLDDT confidence analysis
total_plddt = 3644 + 14584 + 10019 + 581
print(f"Total residues (pLDDT):  {total_plddt}")

28828
